### Setup and Imports
Locate the project root from any notebook working directory, add it to `sys.path`, and import the canonical Phase 1 inspection utilities.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Find the repository root from the current notebook working directory."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "config.yaml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root. Run this notebook inside the project directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import inspect_dataset, load_config, read_csv_sample, resolve_project_path

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
config = load_config(CONFIG_PATH)
DATA_PATH = resolve_project_path(config["paths"]["raw_dataset"], PROJECT_ROOT)
METRICS_DIR = resolve_project_path(config["paths"]["metrics_dir"], PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Metrics directory:", METRICS_DIR)


### Load a Small Sample
Read only the first 100,000 rows for quick visual inspection. Feature decisions are not made from this sequential sample.


In [ ]:
SAMPLE_SIZE = 100_000
sample_df, used_encoding = read_csv_sample(DATA_PATH, SAMPLE_SIZE)

print("Encoding used:", used_encoding)
print("Sample shape:", sample_df.shape)
display(sample_df.head())


### Inspect Columns and Label Candidates
List raw columns and show likely label columns so the selected target can be checked visually.


In [ ]:
label_keywords = ["label", "class", "attack", "category", "type", "target"]
label_candidates = [
    column for column in sample_df.columns
    if any(keyword in column.lower() for keyword in label_keywords)
]

print("Total columns:", len(sample_df.columns))
print("Label candidates:", label_candidates)
for column in label_candidates:
    print("\n" + "=" * 60)
    print("Column:", column)
    print("Unique values in sample:", sample_df[column].nunique(dropna=False))
    print(sample_df[column].value_counts(dropna=False).head(30))


### Run Canonical Phase 1 Inspection
Run the reusable chunked inspection from `src.data_loading`. This scans the full dataset and writes canonical outputs to `results/metrics`.


In [ ]:
inspection_result = inspect_dataset(CONFIG_PATH, sample_size=100_000, chunk_size=100_000)

print("Phase 1 inspection completed.")
for key, value in inspection_result.items():
    print(f"{key}: {value}")


### Review Class Distribution
Load the canonical class distribution generated by the inspection script and verify that it matches the proposal totals.


In [ ]:
class_distribution = pd.read_csv(METRICS_DIR / "class_distribution.csv")

display(class_distribution)
print("Total rows:", int(class_distribution["count"].sum()))
print("Number of classes:", class_distribution.shape[0])


### Review Selected Model Features
Load the canonical candidate feature list. After class-conditional review, this project keeps 69 model features.


In [ ]:
candidate_features = (METRICS_DIR / "candidate_features.txt").read_text(encoding="utf-8").splitlines()

print("Candidate model features:", len(candidate_features))
for index, feature in enumerate(candidate_features, start=1):
    print(f"{index}. {feature}")


### Review Dropped Features
Check excluded columns and their explicit reasons: identifiers, leakage-prone labels, or empirically quasi-constant features.


In [ ]:
dropped_features = pd.read_csv(METRICS_DIR / "dropped_features.csv")

display(dropped_features)
print("Dropped columns:", dropped_features.shape[0])


### Review Quasi-Constant Decisions
Inspect sparse features that need class-conditional review. `Fwd PSH Flags` and `RST Flag Cnt` are retained because they carry minority-class signal.


In [ ]:
quasi_constant_review = pd.read_csv(METRICS_DIR / "quasi_constant_review.csv")

display(quasi_constant_review)


### Inspect Backdoor Signal Features
Show class-conditional statistics for the two sparse features retained after the review.


In [ ]:
class_signal_summary = pd.read_csv(METRICS_DIR / "class_signal_summary.csv")
backdoor_signal = class_signal_summary[
    (class_signal_summary["feature"].isin(["Fwd PSH Flags", "RST Flag Cnt"]))
    & (class_signal_summary["class"] == "Backdoor")
]

display(backdoor_signal)


### Final Phase 1 Checks
Assert the key Phase 1 expectations before moving to preprocessing.


In [ ]:
expected_rows = config["data"]["expected_rows"]
expected_columns = config["data"]["expected_total_columns"]
expected_classes = config["data"]["expected_num_classes"]
expected_features = config["data"]["expected_feature_count"]

assert inspection_result["total_rows"] == expected_rows
assert inspection_result["total_columns"] == expected_columns
assert inspection_result["num_classes"] == expected_classes
assert len(candidate_features) == expected_features
assert "Fwd PSH Flags" in candidate_features
assert "RST Flag Cnt" in candidate_features

print("Phase 1 notebook checks passed.")
